# Channel Verification — Rogala Lab
Use this notebook to visually inspect images and confirm which channel is DAPI, LAMP1, and Raptor.

**Before running the analysis pipeline, verify:**
- C0 → DAPI (nuclei, bright ovals)
- C1 → LAMP1 (lysosomes, scattered puncta)
- C2 → Raptor (more diffuse cytoplasmic + puncta)

If the order is wrong, update `--ch_dapi / --ch_lamp1 / --ch_marker` in `ingest.py`.

In [ ]:
import numpy as np
import tifffile
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
import glob
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
print('Ready')

## 1. Point to your image folder

In [ ]:
# ── EDIT THIS ─────────────────────────────────────────────────────────────────
IMAGE_DIR = "/Users/vaishnavinagesh/Google Drive/My Drive/Confocal/sdtruong_Exp109/Exp109 Images/"

# Expected channel numbers — change if you think they might be different
CH_DAPI   = 0
CH_LAMP1  = 1
CH_MARKER = 2

MARKER_NAME = 'SHMT1'
# ──────────────────────────────────────────────────────────────────────────────

# Find all tif files robustly, including nested folders
image_path = Path(IMAGE_DIR).expanduser()
all_tifs = sorted(str(p) for p in image_path.rglob('*.tif') if p.is_file())

print(f"Found {len(all_tifs)} .tif files")
print(f"First few:")
for f in all_tifs[:5]:
    print(f"  {Path(f).name}")

## 2. Find one complete triplet (C0, C1, C2) to inspect

In [ ]:
import re
from collections import defaultdict

CH_RE = re.compile(r'_C(\d)\.tif$', re.IGNORECASE)

# Group by stem (everything before _C{n}.tif)
stems = defaultdict(dict)
for f in all_tifs:
    m = CH_RE.search(Path(f).name)
    if m:
        ch   = int(m.group(1))
        stem = f[:f.rfind('_C')]
        stems[stem][ch] = f

# Find triplets that have all three channels
triplets = {s: chs for s, chs in stems.items() 
            if all(ch in chs for ch in [CH_DAPI, CH_LAMP1, CH_MARKER])}

print(f"Found {len(triplets)} complete triplets")
print("\nAvailable triplets:")
for i, stem in enumerate(sorted(triplets.keys())):
    print(f"  [{i}] {Path(stem).name}")

if not triplets:
    print("\nNo complete triplets found. Check whether the files use a different naming pattern.")

In [ ]:
# ── Pick which triplet to inspect ─────────────────────────────────────────────
TRIPLET_IDX = 25   # Change this to inspect different conditions
# ──────────────────────────────────────────────────────────────────────────────

stem   = sorted(triplets.keys())[TRIPLET_IDX]
paths  = triplets[stem]

print(f"Inspecting: {Path(stem).name}")
for ch, path in sorted(paths.items()):
    print(f"  C{ch}: {Path(path).name}")

## 3. Load and inspect each channel

In [ ]:
def load_best_z(path):
    """Load TIFF and return the sharpest z-plane (max variance)."""
    stack = tifffile.imread(path).astype(np.float32)
    if stack.ndim == 3:
        if stack.shape[0] == 0:
            raise ValueError(f"{path} contains no z-planes")
        variances = [np.var(stack[z]) for z in range(stack.shape[0])]
        best = np.argmax(variances)
        print(f"  Shape: {stack.shape} | Best z-plane: {best}/{stack.shape[0]}")
        return stack[best]
    if stack.ndim == 2:
        print(f"  Shape: {stack.shape} (2D)")
        return stack
    raise ValueError(f"Unexpected image shape: {stack.shape}")


def norm(img, plow=1, phigh=99.5):
    img = np.asarray(img, dtype=np.float32)
    if img.size == 0:
        raise ValueError("Empty image passed to norm()")
    lo, hi = np.percentile(img, plow), np.percentile(img, phigh)
    return np.clip((img - lo) / (hi - lo + 1e-6), 0, 1)

print("Loading C0:")
c0 = load_best_z(paths[CH_DAPI])
print("Loading C1:")
c1 = load_best_z(paths[CH_LAMP1])
print("Loading C2:")
cm = load_best_z(paths[CH_MARKER])
print("Done")

## 4. View all channels side by side

**What to look for:**
- **DAPI**: bright oval/round shapes = nuclei. Should be very clear nuclear morphology.
- **LAMP1**: scattered bright dots (puncta) throughout the cytoplasm. More concentrated = more lysosomes.
- **Raptor**: mix of diffuse cytoplasmic signal + some brighter puncta.

In [ ]:
# Crop center for faster display
H, W = c0.shape
S = min(512, H, W)
if H <= 0 or W <= 0:
    raise ValueError(f"Loaded image has invalid shape {c0.shape}")

y0 = max(0, H//2 - S)
y1 = min(H, H//2 + S)
x0 = max(0, W//2 - S)
x1 = min(W, W//2 + S)
if y1 <= y0:
    y0, y1 = 0, H
if x1 <= x0:
    x0, x1 = 0, W
sl = np.s_[y0:y1, x0:x1]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle(f'Channel inspection — {Path(stem).name}', fontsize=12, fontweight='bold')

channels = [
    (c0[sl], f'C{CH_DAPI} — Expected: DAPI (nuclei)', 'Blues'),
    (c1[sl], f'C{CH_LAMP1} — Expected: LAMP1 (lysosomes)', 'Greens'),
    (cm[sl], f'C{CH_MARKER} — Expected: {MARKER_NAME}', 'Reds'),
]

for ax, (img, title, cmap) in zip(axes, channels):
    try:
        normed = norm(img)
    except Exception as e:
        print(f"Skipping display for {title}: {e}")
        continue
    ax.imshow(normed, cmap=cmap, vmin=0, vmax=1)
    ax.set_title(title, fontsize=11, fontweight='bold', pad=8)
    ax.axis('off')
    # Intensity stats
    ax.text(0.02, 0.02, 
            f'min={img.min():.0f}  max={img.max():.0f}  mean={img.mean():.0f}',
            transform=ax.transAxes, color='white', fontsize=8,
            bbox=dict(facecolor='black', alpha=0.6, pad=2))

plt.tight_layout()
plt.show()

print()
print("✓ If C0 shows oval nuclei → DAPI assignment correct")
print("✓ If C1 shows scattered bright dots → LAMP1 assignment correct") 
print("✓ If C2 shows diffuse + puncta pattern → Raptor assignment correct")
print()
print("If wrong, swap the CH_DAPI / CH_LAMP1 / CH_MARKER values at the top of Cell 2")

## 5. Zoom into a single cell for detailed inspection

In [ ]:
# Find brightest region in DAPI (likely a cell)
from skimage.filters import gaussian
from skimage.feature import peak_local_max

# Guard against empty or malformed data
if c0.size == 0 or c1.size == 0 or cm.size == 0:
    raise ValueError("One or more channel images are empty; check the input TIFFs.")

# Use a safe crop window around the brightest point
H, W = c0.shape
S = min(128, H, W)
dapi_smooth = gaussian(c0, sigma=5)
peak = np.unravel_index(dapi_smooth.argmax(), dapi_smooth.shape)
py, px = peak

y0 = max(0, py-S)
y1 = min(H, py+S)
x0 = max(0, px-S)
x1 = min(W, px+S)
if y1 <= y0:
    y0, y1 = 0, H
if x1 <= x0:
    x0, x1 = 0, W
cell_sl = np.s_[y0:y1, x0:x1]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle(f'Single cell zoom — centre at ({px}, {py})', fontsize=12, fontweight='bold')

# Row 1: individual channels
for ax, (img, title, cmap) in zip(axes[0], channels):
    try:
        ax.imshow(norm(img[cell_sl]), cmap=cmap, vmin=0, vmax=1)
    except Exception as e:
        print(f"Skipping zoom display for {title}: {e}")
        continue
    ax.set_title(title.split('—')[1].strip(), fontsize=10)
    ax.axis('off')

# Row 2: RGB merge + histogram
rgb = np.zeros((*c0[cell_sl].shape, 3))
rgb[:,:,2] = norm(c0[cell_sl]) * 0.8   # DAPI = blue
rgb[:,:,1] = norm(c1[cell_sl]) * 0.8   # LAMP1 = green  
rgb[:,:,0] = norm(cm[cell_sl]) * 0.8   # Marker = red
axes[1,0].imshow(np.clip(rgb, 0, 1))
axes[1,0].set_title('RGB merge (Blue=DAPI, Green=LAMP1, Red=Raptor)', fontsize=9)
axes[1,0].axis('off')

# Histograms
for ax, (img, label, col) in zip(axes[1,1:], [
    (c1[cell_sl], 'LAMP1 intensity distribution', '#2C7A8C'),
    (cm[cell_sl], f'{MARKER_NAME} intensity distribution', '#C04020'),
]):
    ax.hist(img.flatten(), bins=100, color=col, alpha=0.7, edgecolor='none')
    ax.set_xlabel('Pixel intensity', fontsize=9)
    ax.set_ylabel('Count', fontsize=9)
    ax.set_title(label, fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    # Mark 85th percentile
    p85 = np.percentile(img, 85)
    ax.axvline(p85, color='red', linestyle='--', linewidth=1.5, label=f'85th pct={p85:.0f}')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 6. Compare the same condition across all channels — whole image

In [ ]:
# Show intensity profiles to confirm channel identity
# DAPI should show clear peaks at nucleus locations
# LAMP1 should show scattered smaller peaks (puncta)
# Raptor should show broader, more diffuse profile

# Take a horizontal line through the middle
mid_row = H // 2
x_vals  = np.arange(W)

fig, axes = plt.subplots(3, 1, figsize=(16, 8), sharex=True)
fig.suptitle('Horizontal intensity profile through image centre\n'
             'DAPI = sharp tall peaks (nuclei) | '
             'LAMP1 = many small peaks (lysosomes) | '
             'Raptor = diffuse + some peaks',
             fontsize=10, fontweight='bold')

profiles = [
    (c0[mid_row, :], f'C{CH_DAPI} — DAPI',   '#3A7ABF'),
    (c1[mid_row, :], f'C{CH_LAMP1} — LAMP1',  '#2C7A8C'),
    (cm[mid_row, :], f'C{CH_MARKER} — {MARKER_NAME}', '#C04020'),
]

for ax, (profile, label, color) in zip(axes, profiles):
    ax.plot(x_vals, profile, color=color, linewidth=0.8, alpha=0.9)
    ax.fill_between(x_vals, 0, profile, color=color, alpha=0.15)
    ax.set_ylabel(label, fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

axes[-1].set_xlabel('X pixel position', fontsize=9)
plt.tight_layout()
plt.show()

print("\nInterpretation guide:")
print("  DAPI: should show 3-5 large sharp peaks = nuclei")
print("  LAMP1: should show many small peaks scattered between nuclei")
print("  Raptor: higher baseline with some peaks = diffuse + puncta")

## 7. Check multiple conditions

In [ ]:
# Compare DAPI channel across several conditions to confirm it's consistent
n_show = min(6, len(triplets))
selected = sorted(triplets.keys())[:n_show]

fig, axes = plt.subplots(2, n_show, figsize=(4*n_show, 8))
fig.suptitle(f'C{CH_DAPI} (DAPI) and C{CH_LAMP1} (LAMP1) across conditions',
             fontsize=11, fontweight='bold')

for col, stem in enumerate(selected):
    paths_here = triplets[stem]
    name = Path(stem).name[:35] + '...' if len(Path(stem).name) > 35 else Path(stem).name
    
    # DAPI row
    d = tifffile.imread(paths_here[CH_DAPI]).astype(np.float32)
    d = d[np.argmax([np.var(d[z]) for z in range(d.shape[0])])] if d.ndim==3 else d
    axes[0, col].imshow(norm(d[H//2-256:H//2+256, W//2-256:W//2+256]),
                        cmap='Blues', vmin=0, vmax=1)
    axes[0, col].set_title(name, fontsize=7)
    axes[0, col].axis('off')
    if col == 0:
        axes[0, col].set_ylabel(f'C{CH_DAPI} DAPI', fontsize=9)
    
    # LAMP1 row  
    l = tifffile.imread(paths_here[CH_LAMP1]).astype(np.float32)
    l = l[np.argmax([np.var(l[z]) for z in range(l.shape[0])])] if l.ndim==3 else l
    axes[1, col].imshow(norm(l[H//2-256:H//2+256, W//2-256:W//2+256]),
                        cmap='Greens', vmin=0, vmax=1)
    axes[1, col].axis('off')
    if col == 0:
        axes[1, col].set_ylabel(f'C{CH_LAMP1} LAMP1', fontsize=9)

plt.tight_layout()
plt.show()

print("If DAPI (row 1) shows nuclei consistently across all conditions → C0 = DAPI ✓")
print("If LAMP1 (row 2) shows scattered puncta consistently → C1 = LAMP1 ✓")

## 8. Verdict

In [ ]:
# Fill this in after inspecting the images above

CONFIRMED_CH_DAPI   = 0   # ← update if wrong
CONFIRMED_CH_LAMP1  = 1   # ← update if wrong  
CONFIRMED_CH_MARKER = 2   # ← update if wrong

print("═" * 50)
print("CONFIRMED CHANNEL ASSIGNMENTS:")
print(f"  DAPI   = C{CONFIRMED_CH_DAPI}")
print(f"  LAMP1  = C{CONFIRMED_CH_LAMP1}")
print(f"  Raptor = C{CONFIRMED_CH_MARKER}")
print("═" * 50)
print()
print("Run ingest.py with these values:")
print(f"  python ingest.py \\")
print(f'    --image_dir "{IMAGE_DIR}" \\')
print(f"    --layout flat \\")
print(f"    --ch_dapi {CONFIRMED_CH_DAPI} --ch_lamp1 {CONFIRMED_CH_LAMP1} --ch_marker {CONFIRMED_CH_MARKER} \\")
print(f"    --output ~/Desktop/080326_manifest.tsv")

## Grant Figure

In [ ]:
import sys, os
sys.path.insert(0, os.path.expanduser(
    "~/Desktop/Rogala Lab Images Olga/potential-segmentation-quantizer"))

import numpy as np
import pandas as pd
import tifffile
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from scipy.stats import linregress
import warnings
warnings.filterwarnings('ignore')

from utils import (load_plane, segment_nuclei, detect_puncta_objects,
                   local_median_subtract, object_colocalization, norm)

# ── Config — edit these ───────────────────────────────────────────────────────
# Images for the visual panels (pick one representative field)
C0_PATH = "/Users/vaishnavinagesh/Desktop/071526/DMSO_Starve_Refed/Slide1-Collection1_XY1784179687_Z00_T0_C0.tif"
C1_PATH = "/Users/vaishnavinagesh/Desktop/071526/DMSO_Starve_Refed/Slide1-Collection1_XY1784179687_Z00_T0_C1.tif"
CM_PATH = "/Users/vaishnavinagesh/Desktop/071526/DMSO_Starve_Refed/Slide1-Collection1_XY1784179687_Z00_T0_C2.tif"

# CSV from object_coloc.py — stats are read from here
CSV_PATH = os.path.expanduser("~/Desktop/results_080326/object_coloc_summary.csv")

# Which condition to highlight in the stats panels
# Must match exactly a label in the CSV
CONDITION_FOR_STATS = "AA+ (with amino acids) / DMSO"

CH1_NAME    = 'LAMP1'    # what C1 actually is
MARKER_NAME = 'Raptor'   # what CM actually is
CELL_LINE   = 'HEK293'

# Analysis params (should match what you used to generate the CSV)
PROJECTION      = 'max'
RESIDUAL_FACTOR = 1.0
WINDOW          = 32
LAMP1_PCT       = 75
MARKER_PCT      = 93
PROXIMITY_PX    = 5

OUTPUT_SVG = os.path.expanduser("~/Desktop/grant_coloc_figure.svg")
OUTPUT_PDF = os.path.expanduser("~/Desktop/grant_coloc_figure.pdf")
OUTPUT_PNG = os.path.expanduser("~/Desktop/grant_coloc_figure.png")
# ──────────────────────────────────────────────────────────────────────────────

# ── Load stats from CSV ───────────────────────────────────────────────────────
df = pd.read_csv(CSV_PATH)
print(f"CSV loaded: {len(df)} cells, {df['label'].nunique()} conditions")
print(f"Available conditions:\n  " + "\n  ".join(sorted(df['label'].unique())))

In [ ]:
# ── Extract stats for the chosen condition ────────────────────────────────────
sub = df[df['label'] == CONDITION_FOR_STATS]
if len(sub) == 0:
    print(f"❌ Condition '{CONDITION_FOR_STATS}' not found in CSV")
    print("Available:", df['label'].unique())
else:
    n = len(sub)
    coloc_fracs   = sub['coloc_frac'].values
    n_ref_vals    = sub['n_ref_puncta'].values if 'n_ref_puncta' in sub.columns \
                    else sub['n_marker_puncta'].values
    n_marker_vals = sub['n_marker_puncta'].values

    # Morphology if available
    has_morph = 'marker_mean_area' in sub.columns
    if has_morph:
        marker_area = sub['marker_mean_area'].dropna().values
        ref_area    = sub['ref_mean_area'].dropna().values

    print(f"\nCondition: {CONDITION_FOR_STATS}")
    print(f"  n cells:        {n}")
    print(f"  Coloc frac:     {coloc_fracs.mean()*100:.1f}% ± "
          f"{coloc_fracs.std()/np.sqrt(n)*100:.1f}%")
    print(f"  {CH1_NAME} puncta/cell: {n_ref_vals.mean():.1f}")
    print(f"  {MARKER_NAME} puncta/cell: {n_marker_vals.mean():.1f}")

In [ ]:
# ── Load images and generate visual panels ────────────────────────────────────
print("Loading images...")
c0 = load_plane(C0_PATH, PROJECTION)
c1 = load_plane(C1_PATH, PROJECTION)
cm = load_plane(CM_PATH, PROJECTION)

# Background subtraction
print("Background subtraction...", end=' ', flush=True)
c1_sub = local_median_subtract(c1, window=WINDOW, residual_factor=RESIDUAL_FACTOR)
cm_sub = local_median_subtract(cm, window=WINDOW, residual_factor=RESIDUAL_FACTOR)
print("done")

# Segmentation
print("Segmenting...", end=' ', flush=True)
nuc_labels, cell_labels = segment_nuclei(c0)
print(f"done — {nuc_labels.max()} nuclei")

# Crop a representative region
H, W = c0.shape
S    = 400
cy, cx = H // 2, W // 2
crop = np.s_[cy-S:cy+S, cx-S:cx+S]

c0c = c0[crop]; c1c = c1[crop]; cmc = cm[crop]
c1s = c1_sub[crop]; cms = cm_sub[crop]
nucc = nuc_labels[crop]; cellc = cell_labels[crop]

# Puncta detection on crop for visualisation
cyto_crop = (cellc > 0) & ~(nucc > 0)
ref_lbl,  _ = detect_puncta_objects(c1s, cyto_crop, min_px=5, percentile=LAMP1_PCT)
mrkr_lbl, _ = detect_puncta_objects(cms, cyto_crop, min_px=5, percentile=MARKER_PCT)
ref_mask  = ref_lbl  > 0
mrkr_mask = mrkr_lbl > 0

# Colocalization overlay
from skimage.morphology import dilation, disk
ref_dilated = dilation(ref_mask, disk(PROXIMITY_PX))
overlap = mrkr_mask & ref_dilated

print(f"Crop puncta — {CH1_NAME}: {ref_mask.sum()//10*10}+ px  "
      f"{MARKER_NAME}: {mrkr_mask.sum()//10*10}+ px  "
      f"Colocalized: {overlap.sum()//10*10}+ px")

In [ ]:
# ── Build figure ──────────────────────────────────────────────────────────────
BG  = '#FFFFFF'
fig = plt.figure(figsize=(20, 13), facecolor=BG)
gs  = gridspec.GridSpec(2, 4, figure=fig,
                         hspace=0.1, wspace=0.06,
                         left=0.03, right=0.97,
                         top=0.91, bottom=0.07)

# ── Row 1: Image panels ───────────────────────────────────────────────────────
# Panel 1 — DAPI (nuclei)
ax0 = fig.add_subplot(gs[0, 0])
ax0.imshow(norm(c0c), cmap='Blues', vmin=0, vmax=1, interpolation='bilinear')
ax0.set_title('DAPI\n(Nuclei)', fontsize=13, fontweight='bold',
              color='#1A2C5C', pad=8)
ax0.axis('off')
sb = 80
ax0.plot([15, 15+sb], [c0c.shape[0]-25, c0c.shape[0]-25],
         color='white', linewidth=3)
ax0.text(15+sb//2, c0c.shape[0]-40, '~5 µm', color='white',
         fontsize=9, ha='center')

# Panel 2 — reference channel
ax1 = fig.add_subplot(gs[0, 1])
ax1.imshow(norm(c1c), cmap='Greens', vmin=0, vmax=1, interpolation='bilinear')
ax1.set_title(f'{CH1_NAME}\n(reference channel)', fontsize=13, fontweight='bold',
              color='#1A5C2A', pad=8)
ax1.axis('off')

# Panel 3 — marker channel
ax2 = fig.add_subplot(gs[0, 2])
ax2.imshow(norm(cmc), cmap='Reds', vmin=0, vmax=1, interpolation='bilinear')
ax2.set_title(f'{MARKER_NAME}\n(marker channel)', fontsize=13, fontweight='bold',
              color='#7A1515', pad=8)
ax2.axis('off')

# Panel 4 — colocalization overlay
ax3 = fig.add_subplot(gs[0, 3])
rgb = np.zeros((*c1c.shape, 3))
rgb[:,:,1] = norm(c1c) * 0.85   # green = reference
rgb[:,:,0] = norm(cmc) * 0.85   # red   = marker
# Colocalized pixels = yellow
rgb[overlap, 0] = 1; rgb[overlap, 1] = 1; rgb[overlap, 2] = 0
ax3.imshow(np.clip(rgb, 0, 1), interpolation='bilinear')
ax3.set_title('Overlay\n(Yellow = colocalized)', fontsize=13,
              fontweight='bold', color='#1E2D3A', pad=8)
ax3.legend(handles=[
    mpatches.Patch(facecolor='#00CC44', label=f'{CH1_NAME} only'),
    mpatches.Patch(facecolor='#CC2200', label=f'{MARKER_NAME} only'),
    mpatches.Patch(facecolor='yellow',  label='Colocalized'),
], loc='lower left', fontsize=8, facecolor='black',
   labelcolor='white', framealpha=0.75, edgecolor='none')
ax3.axis('off')

# ── Row 2: Stats from CSV ─────────────────────────────────────────────────────
# Panel 5 — per-cell colocalization bar
ax4 = fig.add_subplot(gs[1, 0])
ax4.set_facecolor('#FFFFFF')
ax4.bar(range(1, len(coloc_fracs)+1), coloc_fracs*100,
        color='#4A9BAF', edgecolor='none', width=0.7)
ax4.axhline(coloc_fracs.mean()*100, color='#D4845A', linewidth=2,
            linestyle='--',
            label=f'Mean: {coloc_fracs.mean()*100:.1f}%')
ax4.set_xlabel('Cell', fontsize=10, color='#1E2D3A')
ax4.set_ylabel(f'% {MARKER_NAME} puncta\ncolocalized with {CH1_NAME}',
               fontsize=10, color='#1E2D3A')
ax4.set_title('Object-Based Colocalization\n(per cell, from CSV)',
              fontsize=11, fontweight='bold', color='#1E2D3A')
ax4.legend(fontsize=10, facecolor='white')
ax4.set_ylim(0, 100)
ax4.tick_params(colors='#1E2D3A')
for sp in ['top','right']: ax4.spines[sp].set_visible(False)
for sp in ['bottom','left']: ax4.spines[sp].set_color('#C0C0C0')

# Panel 6 — all conditions comparison
ax5 = fig.add_subplot(gs[1, 1])
ax5.set_facecolor('#FFFFFF')
cond_stats = (df.groupby('label')['coloc_frac']
                .agg(['mean','std','count'])
                .reset_index()
                .sort_values('mean', ascending=False))
cond_stats['sem'] = cond_stats['std'] / np.sqrt(cond_stats['count'])
x = np.arange(len(cond_stats))
bar_colors = ['#D4845A' if row['label']==CONDITION_FOR_STATS
              else '#4A9BAF'
              for _, row in cond_stats.iterrows()]
ax5.bar(x, cond_stats['mean']*100, color=bar_colors,
        edgecolor='none', width=0.6)
ax5.errorbar(x, cond_stats['mean']*100, yerr=cond_stats['sem']*100,
             fmt='none', ecolor='#1E2D3A', elinewidth=1.5, capsize=4)
xlbls = [l.replace(' / ','\n').replace('amino acids','AA')
         .replace('with','w/').replace('without','w/o')
         for l in cond_stats['label']]
ax5.set_xticks(x)
ax5.set_xticklabels(xlbls, fontsize=6, color='#1E2D3A',
                    rotation=35, ha='right')
ax5.set_ylabel(f'% {MARKER_NAME} on {CH1_NAME}\n(mean ± SEM)',
               fontsize=10, color='#1E2D3A')
ax5.set_title('All Conditions\n(highlighted = selected condition)',
              fontsize=11, fontweight='bold', color='#1E2D3A')
ax5.set_ylim(0, 100)
ax5.tick_params(colors='#1E2D3A')
for sp in ['top','right']: ax5.spines[sp].set_visible(False)
for sp in ['bottom','left']: ax5.spines[sp].set_color('#C0C0C0')

# Panel 7 — puncta counts from CSV
ax6 = fig.add_subplot(gs[1, 2])
ax6.set_facecolor('#FFFFFF')
ref_mean    = n_ref_vals.mean()
ref_sem     = n_ref_vals.std() / np.sqrt(n)
marker_mean = n_marker_vals.mean()
marker_sem  = n_marker_vals.std() / np.sqrt(n)
coloc_mean  = (coloc_fracs * n_marker_vals).mean()

ax6.bar([0, 1, 2],
        [ref_mean, marker_mean, coloc_mean],
        color=['#2C7A8C','#D4845A','#F0C040'],
        edgecolor='none', width=0.5)
ax6.errorbar([0, 1], [ref_mean, marker_mean],
             yerr=[ref_sem, marker_sem],
             fmt='none', ecolor='#1E2D3A', elinewidth=2, capsize=6)
for xi, (v, lbl) in enumerate(zip(
        [ref_mean, marker_mean, coloc_mean],
        [ref_mean, marker_mean, coloc_mean])):
    ax6.text(xi, v + 0.5, f'{v:.1f}', ha='center', va='bottom',
             fontsize=11, fontweight='bold', color='#1E2D3A')
ax6.set_xticks([0, 1, 2])
ax6.set_xticklabels([f'{CH1_NAME}\npuncta/cell',
                     f'{MARKER_NAME}\npuncta/cell',
                     f'Colocalized\npuncta/cell'],
                    fontsize=9, color='#1E2D3A')
ax6.set_ylabel('Mean puncta per cell', fontsize=10, color='#1E2D3A')
ax6.set_title('Puncta Counts\n(from CSV)', fontsize=11,
              fontweight='bold', color='#1E2D3A')
ax6.tick_params(colors='#1E2D3A')
for sp in ['top','right']: ax6.spines[sp].set_visible(False)
for sp in ['bottom','left']: ax6.spines[sp].set_color('#C0C0C0')

# Panel 8 — summary text box
ax7 = fig.add_subplot(gs[1, 3])
ax7.set_facecolor('#FFFFFF'); ax7.axis('off')
summary = (
    f"{CELL_LINE}\n"
    f"{CONDITION_FOR_STATS}\n"
    f"n = {n} cells\n\n"
    f"Object-based colocalization\n"
    f"(mean ± SEM per cell):\n\n"
    f"  % {MARKER_NAME} on {CH1_NAME}\n"
    f"  {coloc_fracs.mean()*100:.1f}% ± "
    f"{coloc_fracs.std()/np.sqrt(n)*100:.1f}%\n\n"
    f"  {CH1_NAME} puncta/cell\n"
    f"  {ref_mean:.1f} ± {ref_sem:.1f}\n\n"
    f"  {MARKER_NAME} puncta/cell\n"
    f"  {marker_mean:.1f} ± {marker_sem:.1f}\n\n"
    f"Methods:\n"
    f"  Max intensity projection\n"
    f"  Local median bg subtraction\n"
    f"  ({WINDOW}×{WINDOW}px, rf={RESIDUAL_FACTOR})\n"
    f"  Nucleus excluded from ROI\n"
    f"  Object-based: centroid\n"
    f"  proximity {PROXIMITY_PX}px\n"
    f"  {CH1_NAME} percentile: {LAMP1_PCT}\n"
    f"  {MARKER_NAME} percentile: {MARKER_PCT}"
)
ax7.text(0.05, 0.97, summary, transform=ax7.transAxes,
         fontsize=9.5, va='top', ha='left', color='#1E2D3A',
         fontfamily='monospace',
         bbox=dict(facecolor='#F5F5F5', edgecolor='#C0C0C0',
                   boxstyle='round,pad=0.8', linewidth=1))
ax7.set_title('Summary Statistics', fontsize=11,
              fontweight='bold', color='#1E2D3A')

# ── Title ─────────────────────────────────────────────────────────────────────
fig.suptitle(f'Colocalization Analysis — {CELL_LINE}  |  '
             f'{CH1_NAME} × {MARKER_NAME}',
             fontsize=14, fontweight='bold', color='#1E2D3A', y=0.97)

plt.savefig(OUTPUT_SVG, format='svg', dpi=300,
            bbox_inches='tight', facecolor=BG)
plt.savefig(OUTPUT_PDF, format='pdf', dpi=300,
            bbox_inches='tight', facecolor=BG)
plt.savefig(OUTPUT_PNG, format='png', dpi=200,
            bbox_inches='tight', facecolor=BG)
plt.show()
print(f"\nSaved:\n  {OUTPUT_SVG}\n  {OUTPUT_PDF}\n  {OUTPUT_PNG}")